# Prepare Austrian Glacier Inventory Ingest Files

The Austrian Glacier Inventory has separate files for each of the 20 regions.  Currently, GLIMS is set up to ingest files by year.  So the 20 GeoJSONS are merged and split into separate years.

We need to add analysts to the people table

In [1]:
# import sys
# sys.path.append("../src")

from pathlib import Path
import re
import json
import datetime as dt

from itertools import chain

import geopandas as gpd
import pandas as pd

from glims_ingest_tools.writers import make_image_gmt, make_session_gmt

from utils import get_images_from_geodataframe

DATAPATH = Path.home() / "Data" / "GLIMS" / "submissions" / "0007_hartl_austria" / "GI5_v2"

SUBMISSON_PATH = DATAPATH / "submission"
FOR_INGEST_PATH = DATAPATH / "for_ingest"

write_file = True  #False

## Define static parameters

Define analysis time.

For AGI5, `analysis_time` we use November 2025 as suggested by Lea Hartl

In [2]:
analysis_time = dt.datetime(2025,11,1)

In [3]:
session_data_source = "Aerial orthoimagery, high-resolution satellite images and Lidar DEMs"
session_process_description = "Manual Digitization"

## Get list of files

The `GI5_v2` directory contains GeoJSON files containing outlines and attributes for each of the 20 regions.

There is also a `documentation` directory that contains a data dictionary and a list of analysts for each region; and a `vanishing_glaciers.geojson`.  This file contains a list of vanished and vanishing glaciers.  It is ignored for this ingest and will be ingested separately.

In [4]:
OUTLINE_FILES = sorted([f for f in SUBMISSON_PATH.glob("*.geojson") if "vanishing" not in f.name])
OUTLINE_FILES

[PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Allgaeuer_Alpen_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Ankogel_Hochalmspitzgruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Dachstein_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Defreggergruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Glocknergruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Goldberggruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Granatspitzgruppe_GI5_proc2.geojson'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/submission/Hochkoenig_GI5_proc2.geojson'),
 PosixPa

## Select Single Filepath

In [5]:
filepath = OUTLINE_FILES[0]  # Set filepath

## Extract Region Name

In [6]:
region = re.search(r'(.+)_GI5_proc2.geojson', filepath.name)  # Get region name
region_name = region.group(1)
print(region_name)

Allgaeuer_Alpen


## Create a sub-directory for the region

In [7]:
REGION_INGEST_PATH = FOR_INGEST_PATH / region_name.lower()
REGION_INGEST_PATH.mkdir(parents=True, exist_ok=True)

In [8]:
list(FOR_INGEST_PATH.glob('*'))

[PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/for_ingest/ankogel_hochalmspitzgruppe'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/for_ingest/allgaeuer_alpen'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/for_ingest/subregions.json'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/for_ingest/metadata_template.json'),
 PosixPath('/home/apbarret/Data/GLIMS/submissions/0007_hartl_austria/GI5_v2/for_ingest/metadata_template.txt')]

## Load Ancillary Files

metadata template

In [9]:
with open(FOR_INGEST_PATH / "metadata_template.json", "r") as f:
    metadata = json.load(f)
# metadata

Subregion metadata - contact and list of analysts

In [10]:
with open(FOR_INGEST_PATH / "subregions.json", "r") as f:
    subregions = json.load(f)

## Extract Region Metadata For Region

Region metadata along with information from the metadata template are used to create `session.gmt`

In [11]:
region_metadata = [subregion for subregion in subregions if subregion["name"] == ' '.join(region_name.split('_'))][0]
region_metadata

{'number': '1',
 'name': 'Allgaeuer Alpen',
 'contact': 'IGF-Glaziologie@oeaw.ac.at',
 'institute': 'Austrian Academy of Sciences, Institute for Interdisciplinary Mountain Research (OEAW-IGF)',
 'analysts': [{'analyst': {'first_name': 'Marie',
    'middle_name': 'Sophie',
    'surname': 'Kaucher'}}]}

## Load GeoJSON for Region

In [12]:
gdf = gpd.read_file(filepath)
gdf.head()
# gdf = pd.concat([gpd.read_file(f) for f in OUTLINE_FILES], ignore_index=True)
# gdf.head()

,id,year,name,data_type,src_date,source,image_ID1,flightstr,state,region,...,source_2,src_date_2,ROGI,van_fragm,nr_frag,median_elev,max_elev,min_elev,area,geometry
0,1800900,2023,Schwarzmilz Ferner,aerial image,20230905,\nPortal Tirol\nLBA\n,40641,0020,Tyrol,Allgaeuer_Alpen,...,,,0,None,1,2429.0,2454.0,2409.0,19941,"POLYGON ((170417.348 381998.047, 170417.041 38..."


In [13]:
dataset_centroid = (
    gpd.GeoSeries([gdf.centroid.union_all().centroid], crs=gdf.crs).to_crs(4326)[0].x,
    gpd.GeoSeries([gdf.centroid.union_all().centroid], crs=gdf.crs).to_crs(4326)[0].y
)                   

## Transform CRS to WGS84
_Need EPSG for lon, lat in WGS84_

In [14]:
gdf = gdf.to_crs(4326)

## Get bounding box of shapefile

In [15]:
shapefile_bounds = gdf.total_bounds
shapefile_bounds

array([10.295218  , 47.29654999, 10.29782424, 47.29842088])

## Homogenize Shapefile `source` names

Lea Hartl provided a text file containing key:value pairs for source name used on original shapefiles and full official names of these sources.  This was processed using `make_source_data_lookup_table.py` to create a JSON file containing a set of unique lookups.  This is used to homogenize source names.

In [16]:
with open('geojson_abbreviations_lookup.json', 'r') as f:
        source_lookup = json.load(f)
# source_lookup

Some edge cases remain.  These are fixed here.  Replace `source` column that contains list with LBA with just LBA

In [17]:
gdf["source"] = gdf["source"].where(gdf.source.str.contains("LBA") == False, "LBA")

Split `source` with two valid sources between `source` and `source_2`

In [18]:
tmp_index = gdf.source == 'Land Salzburg, Land Tirol'
gdf.loc[tmp_index, "source"] = "Land Tirol"
gdf.loc[tmp_index, "source_2"] = "Land Salzburg"

Homogenize source and source 2 using `source_lookup`

In [19]:
gdf["source"] = gdf.source.map(source_lookup)
gdf["source_2"] = gdf.source_2.map(source_lookup)

In [20]:
gdf

,id,year,name,data_type,src_date,source,image_ID1,flightstr,state,region,...,source_2,src_date_2,ROGI,van_fragm,nr_frag,median_elev,max_elev,min_elev,area,geometry
0,1800900,2023,Schwarzmilz Ferner,aerial image,20230905,Orthoimagery Province of Tyrol,40641,0020,Tyrol,Allgaeuer_Alpen,...,NaN,,0,None,1,2429.0,2454.0,2409.0,19941,"POLYGON ((10.29537 47.29704, 10.29536 47.29704..."


## Fix `src_date` format

`src_date` is required to have YYYY-MM-DD format.  `src_date` date formats in shapefile have a mixed format.  Some are `YYYYMMXX`, some are just year (YYYY).

`YYYYMMXX` are set to the first of the month.
`YYYY` is set to Jan 1.

In [21]:
gdf['src_date'] = gdf.src_date.str.replace(r'XX', '01')
gdf['src_date'] = gdf.src_date.where(gdf.src_date.str.match(r'^\d{8}$'), gdf.src_date + '0101')
assert len(gdf) == gdf.src_date.str.match(r'^\d{8}$').sum()  # Assume either correct or two incorrect date formats

## Extract GeoJSON of Images

IDs can be all digits or digit-strings separated by hyphens

Some `image_ID1` are `None`.  These values imputed using `flightstr`, which all entities with missing `image_ID1` have.

In [22]:
images = get_images_from_geodataframe(gdf)

## Create `image.gmt`

In [23]:
image_gmt_str = make_image_gmt(shapefile_bounds, gdf.crs.to_wkt(), images)
if write_file:
    with open(REGION_INGEST_PATH / "image.gmt", "w") as fo:
        fo.write(image_gmt_str)
else:
    print(image_gmt_str)

## Create `session.gmt`

In [24]:
metadata

{'rc_id': '34',
 'region_label': 'Austria',
 'contact_id': '<add Lea Hartl ID>',
 'file_name': '<region shape file>',
 'file_size': '<size of file>',
 'submission_time': '20260409_000000',
 'geocoding_with_image': 'yes',
 'georegistration_gpcs': 'no',
 'orthorectification_dem_dtm': 'yes',
 'dn_radiance_conversion': 'no',
 'sun_elev_correction': 'no',
 'image_radiometric_correction': 'no',
 'model_radiometric_correction': 'no',
 'anisotropic_reflectance': 'no',
 'slope_aspect_correction': 'no',
 'band_ratio_linear_trans': 'no',
 'spatial_filtering': 'no',
 'geomorphological_analysis': 'no',
 'texture_analysis': 'no',
 'manual_digitization': 'yes',
 'percent_manual_editing': 100,
 'supervised_classification': 'no',
 'supervised_title': '',
 'unsupervised_classification': 'no',
 'unsupervised_title': '',
 'process_description': 'Manual digitization of aerial orthoimagery, high resolution satellite images and Lidar DEMs',
 'embargo_period': 'None',
 'analysis_tools': '',
 'analysis_tools_o

In [25]:
session_gmt_str = make_session_gmt(
    shapefile_bounds,
    gdf.crs.to_wkt(),
    metadata["rc_id"],
    analysis_time,
    session_data_source,
    session_process_description,
    [(analyst["analyst"]["surname"], analyst["analyst"]["first_name"]) for analyst in region_metadata["analysts"]],
    "",
    dataset_centroid,
)
if write_file:
    with open(REGION_INGEST_PATH / "session.gmt", "w") as fo:
        fo.write(session_gmt_str)
else:
    print(session_gmt_str)

## Create metadata.txt

In [26]:
from glims_ingest_tools.classes import NewSubmission

In [27]:
new_submission = NewSubmission.from_json(FOR_INGEST_PATH / "metadata_template.json")
new_submission.file_name = filepath.name
new_submission.file_size = filepath.stat().st_size
print(new_submission)

rc_id: '34'
region_label: 'Austria'
contact_id: '<add Lea Hartl ID>'
file_name: 'Allgaeuer_Alpen_GI5_proc2.geojson'
file_size: 6389
submission_time: '20260409_000000'
geocoding_with_image: 'yes'
georegistration_gpcs: 'no'
orthorectification_dem_dtm: 'yes'
dn_radiance_conversion: 'no'
sun_elev_correction: 'no'
image_radiometric_correction: 'no'
model_radiometric_correction: 'no'
anisotropic_reflectance: 'no'
slope_aspect_correction: 'no'
band_ratio_linear_trans: 'no'
spatial_filtering: 'no'
geomorphological_analysis: 'no'
texture_analysis: 'no'
manual_digitization: 'yes'
percent_manual_editing: 100
supervised_classification: 'no'
supervised_title: ''
unsupervised_classification: 'no'
unsupervised_title: ''
process_description: 'Manual digitization of aerial orthoimagery, high resolution satellite images and Lidar DEMs'
embargo_period: 'None'
analysis_tools: ''
analysis_tools_other: ''


In [28]:
if write_file:
    new_submission.write_metadata_file(REGION_INGEST_PATH / "metadata.txt")
else:
    new_submission.print_metadata_file()


## Create glaciers.shp

### Parse Shapefile to format of GLIMS

`glaciers.shp` requires the following fields.

- `loc_unc_x`  The local positional uncertainty of _x_ in m, based on grid cell size
- `loc_unc_y`  The local positional uncertainty of _y_ in m, based on grid cell size
- `glob_unc_x` Positional uncertainty of _x_ in m based on geolocation uncertainty
- `glob_unc_y` Positional uncertainty of _y_ in m based on geolocation uncertainty
- `name`  Name of glacier if one exists, "" if no name exists.

In [29]:
glacier_column_names = {
    'id': 'local_id',         # glacier ID number as per the Austrian glacier inventories
    'name': 'name',           # glacier name as per the Austrian glacier inventories
    'src_date': 'src_date',   # date of image/data acquisition. Example: date of survey flight for aerial imagery
    'area': 'area_m2',        # glacier area in m2, computed for the CRS of the file.
}

In [30]:
from glims_ingest_tools.shapefile import GLACIER_REQUIRED_FIELDS

In [31]:
glacier_gdf = gdf.rename(columns=glacier_column_names)[list(glacier_column_names.values()) + ['geometry', 'source']]
glacier_gdf.head()

,local_id,name,src_date,area_m2,geometry,source
0,1800900,Schwarzmilz Ferner,20230905,19941,"POLYGON ((10.29537 47.29704, 10.29536 47.29704...",Orthoimagery Province of Tyrol


## Add missing required fields and values

Local and global uncertainty fields are missing from the shapefiles.  These are estimated from the paper.

Estimates of local uncertainty in `x` and `y` directions are based on pixel sizes.  We take the upper bound of uncertainty for each source.

Estimates of global uncertainty are based on 2 pixel widths (heights) for each source-type.

|Source-Type|Local Uncertainty (m)|Global Uncertainty (m)|
|-----------|----------|---------|
|Aerial Orthoimagery| 0.10 to 0.25 |0.5|
|DEM Hillshade| 0.5 to 1.0 |2.0|
|Pleiades| 0.5 |1.0|
|Planetscope|3.0|6.0|

Where multiple image sources are used, we assign the uncertainty associated with the largest uncertainty.

In [32]:
uncertainty_lookup = {
    'orthoimagery': {
        'local': 0.25,
        'global': 0.5,
    },
    'pleiades': {
        'local': 0.5,
        'global': 1.0,
    },
    'dem': {
        'local': 1.0,
        'global': 2.0,
    },
    'planetscope': {
        'local': 3.0,
        'global': 6.0,
    },
}   

In [33]:
def source_type(x):
    if (('orthoimage' in x.source.lower()) & 
        (not 'dem' in x.source.lower()) &
        (not 'planetscope' in x.source.lower())):
        return 'orthoimagery'
    elif ('orthoimage' in x.source.lower()) & ('dem' in x.source.lower()):
        return 'dem'
    elif ('dem' in x.source.lower()) & (not 'orthoimage' in x.source.lower()):
        return 'dem'
    elif ('planetscope' in x.source.lower()):
        return 'planetscope'
    elif ('photogrammetric' in x.source.lower()):
        return 'orthoimagery'
    

In [34]:
glacier_gdf['loc_unc_x'] = glacier_gdf.apply(source_type, axis=1).map({k: v['local'] for k, v in uncertainty_lookup.items()}, axis=1)
glacier_gdf['loc_unc_y'] = glacier_gdf.apply(source_type, axis=1).map({k: v['local'] for k, v in uncertainty_lookup.items()}, axis=1)
glacier_gdf['glob_unc_x'] = glacier_gdf.apply(source_type, axis=1).map({k: v['global'] for k, v in uncertainty_lookup.items()}, axis=1)
glacier_gdf['glob_unc_y'] = glacier_gdf.apply(source_type, axis=1).map({k: v['global'] for k, v in uncertainty_lookup.items()}, axis=1)

glacier_gdf.head()

,local_id,name,src_date,area_m2,geometry,source,loc_unc_x,loc_unc_y,glob_unc_x,glob_unc_y
0,1800900,Schwarzmilz Ferner,20230905,19941,"POLYGON ((10.29537 47.29704, 10.29536 47.29704...",Orthoimagery Province of Tyrol,0.25,0.25,0.5,0.5


In [35]:
if write_file:
    glacier_gdf.to_file(REGION_INGEST_PATH / "glaciers.shp")

In [37]:
ls -1 {FOR_INGEST_PATH}

allgaeuer_alpen/
ankogel_hochalmspitzgruppe/
metadata_template.json
metadata_template.txt
subregions.json


In [36]:
ls -1 {REGION_INGEST_PATH}

glaciers.cpg
glaciers.dbf
glaciers.prj
glaciers.shp
glaciers.shx
image.gmt
metadata.txt
session.gmt
